# Notebook 2 : Clause Selction and Risk Taxonomy

This notebook transforms raw legal clauses into a structured business risk taxonomy that powers MeridianIQ's intelligence engine.

While the CUAD dataset tells us whether a clause exists, it does not explain the business impact of that clause. In this notebook, we design the knowledge layer that allows MeridianIQ to interpret legal clauses in terms of operational, financial, contractual, and business risk.

The outputs generated here become the central configuration used throughout the remainder of the project, enabling consistent clause detection, contract fingerprinting, risk scoring, executive reporting, and LLM orchestration.

# Loading Libraries and Dataset

In [ ]:
import pandas as pd
import numpy as np
import json
from column_groups import *

pd.set_option("display.max_columns",None)
pd.set_option("display.max_colwidth",200)

Context Columns:  41
Answer Columns:  41
Binary Columns:  33
Entity Columns:  8


In [ ]:
df=pd.read_csv("master_clauses.csv")

In [ ]:
df.shape

(510, 83)

In [ ]:
clause_table=pd.DataFrame({
    "answer_column":binary_cols
})

clause_table["clause_name"]=(
    clause_table["answer_column"]
    .str.replace("-Answer","")
    .str.replace("- Answer","")
    .str.strip()
)

clause_table["context_column"]=clause_table["clause_name"]
clause_table.head()

,answer_column,clause_name,context_column
0,Most Favored Nation-Answer,Most Favored Nation,Most Favored Nation
1,Competitive Restriction Exception-Answer,Competitive Restriction Exception,Competitive Restriction Exception
2,Non-Compete-Answer,Non-Compete,Non-Compete
3,Exclusivity-Answer,Exclusivity,Exclusivity
4,No-Solicit Of Customers-Answer,No-Solicit Of Customers,No-Solicit Of Customers


##Constructing the Clause Metadata Table

Before assigning business meaning to clauses, we first build a structured metadata table containing every legal clause available in the CUAD dataset.

Instead of manually maintaining clause names throughout the project, this table becomes a single source of truth that every subsequent notebook can reference.

In [ ]:
clause_table=clause_table.merge(
    presence_df,
    on="clause_name",
    how="left"
)

clause_table.sort_values("presence_rate",ascending=False).head(10)

,answer_column,clause_name,context_column,yes_count,no_count,presence_rate
10,Anti-Assignment-Answer,Anti-Assignment,Anti-Assignment,374,136,0.733333
27,Cap On Liability-Answer,Cap On Liability,Cap On Liability,275,235,0.539216
17,License Grant-Answer,License Grant,License Grant,255,255,0.500000
25,Audit Rights-Answer,Audit Rights,Audit Rights,214,296,0.419608
7,Termination For Convenience-Answer,Termination For Convenience,Termination For Convenience,183,327,0.358824
24,Post-Termination Services-Answer,Post-Termination Services,Post-Termination Services,182,328,0.356863
3,Exclusivity-Answer,Exclusivity,Exclusivity,180,330,0.352941
30,Insurance-Answer,Insurance,Insurance,167,343,0.327451
11,Revenue/Profit Sharing-Answer,Revenue/Profit Sharing,Revenue/Profit Sharing,166,344,0.325490
13,Minimum Commitment-Answer,Minimum Commitment,Minimum Commitment,165,345,0.323529


##Assigning Risk Domains

Legal clauses rarely exist in isolation.Many clauses protect against similar business concerns even if they describe different legal obligations.

In this step, every clause is grouped into a broader business-oriented risk domain.

This taxonomy later supports executive dashboards, risk summaries, and LLM-generated reports.

In [ ]:
risk_domain_map={

    #Liability Risk
    "Uncapped Liability":"Liability Risk",
    "Cap On Liability":"Liability Risk",
    "Insurance":"Liability Risk",
    "Liquidated Damages":"Liability Risk",

    #Competition Risk
    "Non-Compete":"Competition Risk",
    "Exclusivity":"Competition Risk",
    "No-Solicit Of Employees":"Competition Risk",
    "Competitive Restriction Exception":"Competition Risk",
    "Non-Disparagement":"Competition Risk",
    "No-Solicit Of Customers":"Competition Risk",


    # Control / Transfer Risk
    "Anti-Assignment":"Control Risk",
    "Change Of Control":"Control Risk",
    "Rofr/Rofo/Rofn":"Control Risk",
    "Third Party Beneficiary":"Control Risk",

    # Financial / Commercial Risk
    "Revenue/Profit Sharing":"Financial Risk",
    "Price Restrictions":"Financial Risk",
    "Minimum Commitment":"Financial Risk",
    "Volume Restriction":"Financial Risk",
    "Most Favored Nation":"Financial Risk",

    # IP / Licensing Risk
    "Ip Ownership Assignment":"IP Risk",
    "Joint Ip Ownership":"IP Risk",
    "License Grant":"IP Risk",
    "Non-Transferable License":"IP Risk",
    "Affiliate License-Licensor":"IP Risk",
    "Affiliate License-Licensee":"IP Risk",
    "Unlimited/All-You-Can-Eat-License":"IP Risk",
    "Irrevocable Or Perpetual License":"IP Risk",
    "Source Code Escrow":"IP Risk",
    "Covenant Not To Sue":"IP Risk",

    # Operational / Lifecycle Risk
    "Termination For Convenience":"Lifecycle Risk",
    "Post-Termination Services":"Lifecycle Risk",
    "Audit Rights":"Operational Risk",
    "Warranty Duration":"Operational Risk"

}

clause_table["risk_domain"]=clause_table["clause_name"].map(risk_domain_map)
clause_table[clause_table["risk_domain"].isna()]


,answer_column,clause_name,context_column,yes_count,no_count,presence_rate,risk_domain


##Defining the Nature of Each Clause

Not every legal clause should increase risk simply because it exists.

Some clauses reduce risk by protecting the contracting parties, while others increase contractual exposure when present.

To capture this distinction, every clause is assigned one of three roles:

1.Risk if Present

2.Risk if Missing

3.Contextual

This classification forms the foundation of MeridianIQ's decision engine and determines how individual clauses contribute to overall contract risk.

In [ ]:
clause_role_map={

    #Risk if present
    "Uncapped Liability":"Risk if Present",
    "Non-Compete":"Risk if Present",
    "Exclusivity":"Risk if Present",
    "Minimum Commitment":"Risk if Present",
    "Revenue/Profit Sharing":"Risk if Present",
    "Most Favored Nation":"Risk if Present",
    "Price Restrictions":"Risk if Present",
    "Volume Restriction":"Risk if Present",
    "Liquidated Damages":"Risk if Present",
    "Irrevocable Or Perpetual License":"Risk if Present",
    "No-Solicit Of Employees":"Risk if Present",
    "Unlimited/All-You-Can-Eat-License":"Risk if Present",
    "Post-Termination Services":"Risk if Present",
    "No-Solicit Of Customers":"Risk if Present",
    "No-Solicit Of Employees":"Risk if Present",
    "Non-Disparagement":"Risk if Present",

    # Risk if missing
    "Cap On Liability":"Risk if Missing",
    "Insurance":"Risk if Missing",
    "Termination For Convenience":"Risk if Missing",

    # Contextual / review
    "Anti-Assignment":"Contextual",
    "Change Of Control":"Contextual",
    "Audit Rights":"Contextual",
    "Ip Ownership Assignment":"Contextual",
    "Joint Ip Ownership":"Contextual",
    "License Grant":"Contextual",
    "Non-Transferable License":"Contextual",
    "Affiliate License-Licensor":"Contextual",
    "Affiliate License-Licensee":"Contextual",
    "Source Code Escrow":"Contextual",
    "Covenant Not To Sue":"Contextual",
    "Competitive Restriction Exception":"Contextual",
    "Rofr/Rofo/Rofn":"Contextual",
    "Third Party Beneficiary":"Contextual",
    "Warranty Duration":"Contextual"
}



clause_table["clause_role"]=clause_table["clause_name"].map(clause_role_map)
clause_table[clause_table["clause_role"].isna()]



,answer_column,clause_name,context_column,yes_count,no_count,presence_rate,risk_domain,clause_role


##Assigning Severity Levels

Not all legal clauses carry the same business importance.

For example, an uncapped liability clause generally represents greater financial exposure than a marketing restriction.

Each clause is therefore assigned a severity level based on its potential business impact.



In [ ]:
severity_map={
    #Critical
    "Uncapped Liability":"Critical",
    "Cap On Liability":"Critical",
    "Insurance":"Critical",
    "Ip Ownership Assignment":"Critical",
    "Non-Compete":"Critical",
    "Exclusivity":"Critical",

     # High
    "Termination For Convenience":"High",
    "Change Of Control":"High",
    "Minimum Commitment":"High",
    "Revenue/Profit Sharing":"High",
    "Post-Termination Services":"High",
    "Irrevocable Or Perpetual License":"High",
    "Unlimited/All-You-Can-Eat-License":"High",

    # Medium
    "Anti-Assignment":"Medium",
    "Audit Rights":"Medium",
    "No-Solicit Of Customers":"Medium",
    "No-Solicit Of Employees":"Medium",
    "Price Restrictions":"Medium",
    "Volume Restriction":"Medium",
    "Liquidated Damages":"Medium",
    "Rofr/Rofo/Rofn":"Medium",
    "Most Favored Nation":"Medium",
    "Joint Ip Ownership":"Medium",
    "Source Code Escrow":"Medium",

    # Low / Contextual
    "License Grant":"Low",
    "Non-Transferable License":"Low",
    "Affiliate License-Licensor":"Low",
    "Affiliate License-Licensee":"Low",
    "Competitive Restriction Exception":"Low",
    "Non-Disparagement":"Low",
    "Covenant Not To Sue":"Low",
    "Third Party Beneficiary":"Low",
    "Warranty Duration":"Low"
}

clause_table["severity"]=clause_table["clause_name"].map(severity_map)
clause_table[clause_table["severity"].isna()]

,answer_column,clause_name,context_column,yes_count,no_count,presence_rate,risk_domain,clause_role,severity


##Defining Initial Risk Weights

To convert qualitative risk into measurable intelligence, each severity level is mapped to an initial numerical weight.

These weights provide the first version of MeridianIQ's risk scoring mechanism.

Although simple, this approach creates a transparent and explainable baseline that can later be refined using historical contracts or domain expertise.

In [ ]:
severity_weight_map={
    "Critical":25,
    "High":18,
    "Medium":10,
    "Low":5
}

clause_table["base_risk_weight"]=clause_table["severity"].map(severity_weight_map)
clause_table[
    ["clause_name","risk_domain","clause_role","severity","base_risk_weight","presence_rate"]
].sort_values("base_risk_weight",ascending=False)


,clause_name,risk_domain,clause_role,severity,base_risk_weight,presence_rate
2,Non-Compete,Competition Risk,Risk if present,Critical,25,0.233333
30,Insurance,Liability Risk,Risk if Missing,Critical,25,0.327451
3,Exclusivity,Competition Risk,Risk if Present,Critical,25,0.352941
15,Ip Ownership Assignment,IP Risk,Contextual,Critical,25,0.243137
27,Cap On Liability,Liability Risk,Risk if Missing,Critical,25,0.539216
26,Uncapped Liability,Libility Risk,Risk if present,Critical,25,0.217647
22,Irrevocable Or Perpetual License,IP Risk,Risk if present,High,18,0.137255
11,Revenue/Profit Sharing,Financial Risk,Risk if present,High,18,0.325490
7,Termination For Convenience,Lifecycle Risk,Risk if Missing,High,18,0.358824
9,Change Of Control,Control Risk,Contextual,High,18,0.237255


##Selecting the MVP Clauses

The CUAD dataset contains a large number of legal clauses.

However, not every clause is equally important for the first version of MeridianIQ.

This step selects the subset of clauses that provide the highest practical business value while keeping the initial system manageable.

The selected MVP clauses become the foundation for:

Clause Detection
Contract Fingerprinting
Evidence Retrieval
Risk Scoring
Executive Reporting

Future versions of MeridianIQ can easily expand this configuration without redesigning the overall architecture.

In [ ]:
mvp_clauses=[
    #Liability
    "Uncapped Liability",
    "Cap On Liability",
    "Insurance",

    #Competition
    "Non-Compete",
    "Exclusivity",
    "No-Solicit Of Customers",
    "No-Solicit Of Employees",

    # Lifecycle
    "Termination For Convenience",
    "Post-Termination Services",

    # Control
    "Anti-Assignment",
    "Change Of Control",

    # Financial
    "Minimum Commitment",
    "Revenue/Profit Sharing",
    "Price Restrictions",
    "Volume Restriction",

    # IP
    "Ip Ownership Assignment",
    "Joint Ip Ownership",
    "License Grant",
    "Irrevocable Or Perpetual License",
    "Unlimited/All-You-Can-Eat-License",

    # Operational
    "Audit Rights",
    "Warranty Duration",
    "Liquidated Damages"

]

clause_table["is_mvp_clause"]=clause_table["clause_name"].isin(mvp_clauses)
clause_table["is_mvp_clause"].value_counts()

,count
is_mvp_clause,
True,23
False,10


In [ ]:
mvp_clause_table=clause_table[clause_table["is_mvp_clause"]].copy()

mvp_clause_table.sort_values(
    ["risk_domain","base_risk_weight"],
    ascending=[True,False]

)

,answer_column,clause_name,context_column,yes_count,no_count,presence_rate,risk_domain,clause_role,severity,base_risk_weight,is_mvp_clause
2,Non-Compete-Answer,Non-Compete,Non-Compete,119,391,0.233333,Competition Risk,Risk if present,Critical,25,True
3,Exclusivity-Answer,Exclusivity,Exclusivity,180,330,0.352941,Competition Risk,Risk if Present,Critical,25,True
4,No-Solicit Of Customers-Answer,No-Solicit Of Customers,No-Solicit Of Customers,34,476,0.066667,Competition Risk,Risk if Present,Medium,10,True
5,No-Solicit Of Employees-Answer,No-Solicit Of Employees,No-Solicit Of Employees,59,451,0.115686,Competition Risk,Risk if Present,Medium,10,True
9,Change Of Control-Answer,Change Of Control,Change Of Control,121,389,0.237255,Control Risk,Contextual,High,18,True
10,Anti-Assignment-Answer,Anti-Assignment,Anti-Assignment,374,136,0.733333,Control Risk,Contextual,Medium,10,True
11,Revenue/Profit Sharing-Answer,Revenue/Profit Sharing,Revenue/Profit Sharing,166,344,0.325490,Financial Risk,Risk if present,High,18,True
13,Minimum Commitment-Answer,Minimum Commitment,Minimum Commitment,165,345,0.323529,Financial Risk,Risk if present,High,18,True
12,Price Restrictions-Answer,Price Restrictions,Price Restrictions,15,495,0.029412,Financial Risk,Risk if present,Medium,10,True
14,Volume Restriction-Answer,Volume Restriction,Volume Restriction,82,428,0.160784,Financial Risk,Risk if present,Medium,10,True


# Standardizing Risk Domains

In [ ]:
risk_config["risk_domain"] = risk_config["risk_domain"].replace({
    "Libility Risk": "Liability Risk"
})

##Adding Business Descriptions

Legal clause names are often meaningful only to legal professionals.

To make MeridianIQ more accessible to business users, each clause is paired with a concise business description explaining why that clause matters.

This step effectively bridges legal terminology with business language.

In [ ]:
business_description_map = {
    "Uncapped Liability":"Indicates whether a party may face liability without a contractual cap.",
    "Cap On Liability":"Indicates whether the contract limits the maximum liability exposure.",
    "Insurance":"Indicates whether a party must maintain insurance coverage.",
    "Non-Compete":"Restricts a party from competing in certain markets, geographies, or sectors.",
    "Exclusivity":"Creates exclusive dealing obligations that may limit business flexibility.",
    "No-Solicit Of Customers":"Restricts solicitation of the counterparty’s customers or partners.",
    "No-Solicit Of Employees":"Restricts hiring or solicitation of the counterparty’s employees.",
    "Termination For Convenience":"Allows termination without cause after notice or waiting period.",
    "Post-Termination Services":"Creates obligations that continue after contract termination.",
    "Anti-Assignment":"Restricts transfer or assignment of contractual rights.",
    "Change Of Control":"Creates rights or obligations triggered by ownership or control changes.",
    "Minimum Commitment":"Requires minimum purchase, order, usage, or payment commitments.",
    "Revenue/Profit Sharing":"Requires sharing revenue or profit with the counterparty.",
    "Price Restrictions":"Restricts the ability to change prices.",
    "Volume Restriction":"Creates restrictions or penalties based on usage or volume thresholds.",
    "Ip Ownership Assignment":"Transfers ownership of created intellectual property to another party.",
    "Joint Ip Ownership":"Creates shared ownership of intellectual property.",
    "License Grant":"Grants rights to use intellectual property, technology, content, or services.",
    "Irrevocable Or Perpetual License":"Creates license rights that may survive indefinitely or cannot be revoked.",
    "Unlimited/All-You-Can-Eat-License":"Grants broad or unlimited usage rights.",
    "Audit Rights":"Allows one party to inspect records, books, systems, or compliance evidence.",
    "Warranty Duration":"Specifies how long warranties remain valid.",
    "Liquidated Damages":"Defines preset damages or fees for breach or termination."
}


In [ ]:
clause_table["business_description"]=clause_table["clause_name"].map(business_description_map)
clause_table[clause_table["is_mvp_clause"] & clause_table["business_description"].isna()]

,answer_column,clause_name,context_column,yes_count,no_count,presence_rate,risk_domain,clause_role,severity,base_risk_weight,is_mvp_clause,business_description


#Defining Risk Actions

Detecting risk is only part of the problem.

MeridianIQ should also communicate what a reviewer should do after identifying a risky clause.

Each clause is therefore assigned a recommended risk action, such as reviewing contractual obligations, verifying ownership rights, or assessing financial exposure.

In [ ]:
risk_action_map={
    "Risk if Present":"Flag when present",
    "Risk if missing":"Flag when missing",
    "Contextual":"Show for review"
}

clause_table["risk_action"]=clause_table["clause_role"].map(risk_action_map)

##Exporting the Risk Configuration

After combining every manually engineered component, the complete clause taxonomy is exported as a reusable configuration file.

Rather than recreating mappings inside every notebook, MeridianIQ simply loads this configuration whenever business intelligence is required.

In [ ]:
output_cols=[
    "clause_name",
    "context_column",
    "answer_column",
    "risk_domain",
    "clause_role",
    "severity",
    "base_risk_weight",
    "risk_action",
    "presence_rate",
    "is_mvp_clause",
    "business_description",

]

risk_config=clause_table[output_cols].copy()
risk_config.head()

,clause_name,context_column,answer_column,risk_domain,clause_role,severity,base_risk_weight,risk_action,presence_rate,is_mvp_clause,business_description
0,Most Favored Nation,Most Favored Nation,Most Favored Nation-Answer,Financial Risk,Risk if present,Medium,10,NaN,0.054902,False,NaN
1,Competitive Restriction Exception,Competitive Restriction Exception,Competitive Restriction Exception-Answer,Competition Risk,Contextual,Low,5,Show for review,0.149020,False,NaN
2,Non-Compete,Non-Compete,Non-Compete-Answer,Competition Risk,Risk if present,Critical,25,NaN,0.233333,True,"Restricts a party from competing in certain markets, geographies, or sectors."
3,Exclusivity,Exclusivity,Exclusivity-Answer,Competition Risk,Risk if Present,Critical,25,Flag when present,0.352941,True,Creates exclusive dealing obligations that may limit business flexibility.
4,No-Solicit Of Customers,No-Solicit Of Customers,No-Solicit Of Customers-Answer,Competition Risk,Risk if Present,Medium,10,Flag when present,0.066667,True,Restricts solicitation of the counterparty’s customers or partners.


In [ ]:
risk_config.to_csv("clause_risk_config.csv",index=False)
risk_config.to_json("clause_risk_config.json",orient="records",indent=2)

# Verifying the Configuration

In [ ]:
loaded_config=pd.read_csv("clause_risk_config.csv")
loaded_config.shape

(33, 11)

In [ ]:
loaded_config.head()

,clause_name,context_column,answer_column,risk_domain,clause_role,severity,base_risk_weight,risk_action,presence_rate,is_mvp_clause,business_description
0,Most Favored Nation,Most Favored Nation,Most Favored Nation-Answer,Financial Risk,Risk if present,Medium,10,NaN,0.054902,False,NaN
1,Competitive Restriction Exception,Competitive Restriction Exception,Competitive Restriction Exception-Answer,Competition Risk,Contextual,Low,5,Show for review,0.149020,False,NaN
2,Non-Compete,Non-Compete,Non-Compete-Answer,Competition Risk,Risk if present,Critical,25,NaN,0.233333,True,"Restricts a party from competing in certain markets, geographies, or sectors."
3,Exclusivity,Exclusivity,Exclusivity-Answer,Competition Risk,Risk if Present,Critical,25,Flag when present,0.352941,True,Creates exclusive dealing obligations that may limit business flexibility.
4,No-Solicit Of Customers,No-Solicit Of Customers,No-Solicit Of Customers-Answer,Competition Risk,Risk if Present,Medium,10,Flag when present,0.066667,True,Restricts solicitation of the counterparty’s customers or partners.


##Notebook Summary

In this notebook, we transformed raw legal clauses into a structured business intelligence layer.

The resulting risk configuration defines:

Business risk domains
Clause behavior
Severity levels
Initial risk weights
MVP clause selection
Business descriptions
Recommended risk actions

Rather than being hardcoded throughout the project, this configuration becomes MeridianIQ's central knowledge base and serves as the foundation for contract fingerprinting, clause detection, semantic retrieval, risk scoring, and LLM-powered contract intelligence.